
# Multi-Component System

Create complex systems with multiple components:


In [ ]:
import datetime
import os

from pylimer_tools_cpp import DataFileWriter, MCUniverseGenerator

# System parameters
box_size = 60.0
seed = 42
bead_distance = 1.0

# Network parameters
n_crosslinkers = 50

n_polymer_chains = 80
beads_per_chain = 25

n_solvent_chains = 200
beads_per_solvent = 5

n_monofunctional_chains = 50
beads_per_monofunctional_chain = 10

crosslinker_conversion = 0.85
crosslinker_functionality = 4

print(f"Starting network generation at {datetime.datetime.now()}")

# Initialize generator
generator = MCUniverseGenerator(box_size, box_size, box_size)
generator.set_seed(seed)
generator.set_bead_distance(bead_distance)

# Add components in order
print("Adding crosslinkers...")
generator.add_crosslinkers(
    n_crosslinkers,
    crosslinker_functionality=crosslinker_functionality,
    crosslinker_type=2,
)  # Type 2

print("Adding solvent chains...")
generator.add_solvent_chains(n_solvent_chains, beads_per_solvent, 3)  # Type 3

print("Adding and linking polymer strands...")
# Add bifunctional polymer chains
generator.add_strands(
    nr_of_strands=n_polymer_chains,
    strand_lengths=[beads_per_chain for _ in range(n_polymer_chains)],
    strand_atom_type=1,  # Type 1 for "normal" beads
)

generator.add_monofunctional_strands(
    nr_of_monofunctional_strands=n_monofunctional_chains,
    monofunctional_strand_length=[
        beads_per_monofunctional_chain for _ in range(n_monofunctional_chains)
    ],
    # Type 4 to recognize these as "monofunctional" strands later
    monofunctional_strand_atom_type=4,
)

# Crosslink polymer strands
generator.link_strands_to_conversion(
    crosslinker_conversion=crosslinker_conversion,  # 92.5% of crosslinker sites used
)

# Get final system
universe = generator.get_universe()
universe.set_masses({1: 1.0, 2: 2.0, 3: 0.5, 4: 1.0})  # Different masses

print(f"Network generation completed at {datetime.datetime.now()}")

# Save the universe to a file
writer = DataFileWriter(universe)

if not os.path.exists("generated_structures"):
    os.makedirs("generated_structures")

writer.write_to_file(
    "generated_structures/multi_component_polymer_network.data")

## Component Composition Visualization
We plot the distribution of atoms across all network components:



In [ ]:
import matplotlib.pyplot as plt

type_labels = {
    1: "Polymer Strands\n(Type 1)",
    2: "Crosslinkers\n(Type 2)",
    3: "Solvent Beads\n(Type 3)",
    4: "Monofunctional\n(Type 4)",
}
counts = {
    type_labels[t]: len(universe.get_atoms_by_type(t))
    for t in sorted(type_labels.keys())
}

plt.figure(figsize=(8, 4.5))
bars = plt.bar(
    counts.keys(),
    counts.values(),
    color=["#4C72B0", "#DD8452", "#55A868", "#C44E52"],
    edgecolor="black",
    linewidth=0.8,
)
for bar in bars:
    yval = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2.0,
        yval + 15,
        f"{yval:,}",
        ha="center",
        va="bottom",
        fontsize=10,
    )

plt.ylabel("Number of Atoms", fontsize=11)
plt.title("Atom Distribution Across Network Components", fontsize=12)
plt.grid(True, axis="y", linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()